In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv

!command -v mpirun >/dev/null || (apt-get update -qq && apt-get install -y -qq openmpi-bin libopenmpi-dev)

!pip install -q mpi4py numba numpy

!mpirun --version | head -n 2
!lscpu | grep -E 'Model name|CPU\(s\)|cache' || true

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-e32b3777-b8b8-221a-94d6-777193bbe2c5, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6

CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
L1d cache:                               32 KiB (1 instance)
L1i cache:                               32 KiB (1 instance)
L2 cache:                                1 MiB (1 instance)
L3 cache:                                38.5 MiB (1 instance)
NUMA node0 CPU(s):                       0,1


In [2]:
GITHUB_USERNAME = "Beknur0"
print(f"GitHub username: {GITHUB_USERNAME}")

GitHub username: Beknur0


## Task 1 — Hybrid MPI Cartesian Grid and Shared-Memory Window

In [3]:
%%writefile hybrid_mpi_cuda_cache.py
import hashlib
import os
import platform
import subprocess
import sys

import numpy as np
from mpi4py import MPI
from numba import cuda


def get_gpu_uuid():
    output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        text=True,
    )
    return output.strip().splitlines()[0].strip()


def get_cpu_signature():
    model = "unknown"
    try:
        with open("/proc/cpuinfo", encoding="utf-8") as file:
            for line in file:
                if line.startswith("model name"):
                    model = line.split(":", 1)[1].strip()
                    break
    except OSError:
        pass

    return f"{platform.machine()}::{model}::{os.cpu_count()}"


def init_hybrid_context(github_user):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()

    if comm.Get_size() != 4:
        raise RuntimeError("This program requires exactly 4 MPI ranks.")

    if not github_user.strip():
        raise ValueError("GitHub username must not be empty.")

    cuda.select_device(0)

    cart_comm = comm.Create_cart(
        dims=[2, 2],
        periods=[False, False],
        reorder=False,
    )

    up, down = cart_comm.Shift(0, 1)
    left, right = cart_comm.Shift(1, 1)

    node_comm = comm.Split_type(MPI.COMM_TYPE_SHARED)
    node_rank = node_comm.Get_rank()

    if node_comm.Get_size() != 4:
        raise RuntimeError("All 4 ranks must run on the same node.")

    gpu_uuid = get_gpu_uuid()
    cpu_signature = get_cpu_signature()

    identity = f"{github_user.strip()}::{cpu_signature}::{gpu_uuid}"
    digest = hashlib.sha256(identity.encode("utf-8")).digest()
    instance_seed = int.from_bytes(digest[:8], byteorder="big")

    # Allocate extra bytes so the 64-slot view can be aligned.
    nbytes = 64 * np.dtype(np.uint64).itemsize
    allocation_size = nbytes + 63 if node_rank == 0 else 0

    win = MPI.Win.Allocate_shared(
        allocation_size,
        disp_unit=1,
        comm=node_comm,
    )
    shared_buffer, disp_unit = win.Shared_query(0)

    if disp_unit != 1:
        raise RuntimeError("Unexpected shared window displacement unit.")

    raw = np.ndarray(
        shape=(nbytes + 63,),
        dtype=np.uint8,
        buffer=shared_buffer,
    )

    local_offset = (-raw.ctypes.data) % 64
    offsets = node_comm.allgather(local_offset)
    if len(set(offsets)) != 1:
        raise RuntimeError("Shared mappings have incompatible alignment.")

    shm_view = np.ndarray(
        shape=(64,),
        dtype=np.uint64,
        buffer=shared_buffer,
        offset=local_offset,
    )

    if shm_view.ctypes.data % 64 != 0:
        raise RuntimeError("Shared memory view is not 64-byte aligned.")

    if node_rank == 0:
        shm_view.fill(0)

    win.Sync()
    node_comm.Barrier()
    win.Sync()

    return {
        "comm": comm,
        "rank": rank,
        "cart_comm": cart_comm,
        "coords": cart_comm.Get_coords(rank),
        "neighbors": {
            "up": up,
            "down": down,
            "left": left,
            "right": right,
        },
        "node_comm": node_comm,
        "node_rank": node_rank,
        "win": win,
        "shm_view": shm_view,
        "gpu_uuid": gpu_uuid,
        "cpu_signature": cpu_signature,
        "instance_seed": instance_seed,
    }


def main():
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python3 hybrid_mpi_cuda_cache.py GITHUB_USERNAME")

    ctx = init_hybrid_context(sys.argv[1])

    try:
        for rank in range(4):
            ctx["comm"].Barrier()
            if ctx["rank"] == rank:
                print(
                    f"rank={rank}, coords={ctx['coords']}, "
                    f"neighbors={ctx['neighbors']}, "
                    f"shared_address_mod_64={ctx['shm_view'].ctypes.data % 64}",
                    flush=True,
                )

        ctx["comm"].Barrier()
        if ctx["rank"] == 0:
            print(f"GPU UUID: {ctx['gpu_uuid']}", flush=True)
            print(f"CPU signature: {ctx['cpu_signature']}", flush=True)
            print(f"Instance seed: {ctx['instance_seed']}", flush=True)
            print("Hybrid context initialized successfully.", flush=True)
    finally:
        ctx["win"].Free()
        ctx["node_comm"].Free()
        ctx["cart_comm"].Free()


if __name__ == "__main__":
    main()

Overwriting hybrid_mpi_cuda_cache.py


In [4]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "hybrid_mpi_cuda_cache.py",
    GITHUB_USERNAME,
]

proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Execution failed with code {proc.returncode}")

rank=0, coords=[0, 0], neighbors={'up': -2, 'down': 2, 'left': -2, 'right': 1}, shared_address_mod_64=0
rank=1, coords=[0, 1], neighbors={'up': -2, 'down': 3, 'left': 0, 'right': -2}, shared_address_mod_64=0
rank=2, coords=[1, 0], neighbors={'up': 0, 'down': -2, 'left': -2, 'right': 3}, shared_address_mod_64=0
rank=3, coords=[1, 1], neighbors={'up': 1, 'down': -2, 'left': 2, 'right': -2}, shared_address_mod_64=0
GPU UUID: GPU-e32b3777-b8b8-221a-94d6-777193bbe2c5
CPU signature: x86_64::Intel(R) Xeon(R) CPU @ 2.00GHz::2
Instance seed: 17189560884387096906
Hybrid context initialized successfully.



## Task 2 — Rank-Local Fields and Halo Pack/Unpack Kernels

In [5]:
task2_code = '''
LOCAL_N = 256
PADDED_N = LOCAL_N + 2
GLOBAL_N = 512

# Buffer directions: 0 = up, 1 = down, 2 = left, 3 = right.


def initialize_fields(ctx):
    row, col = ctx["coords"]

    global_rows = row * LOCAL_N + np.arange(PADDED_N)
    global_cols = col * LOCAL_N + np.arange(PADDED_N)

    dx = 1.0 / (GLOBAL_N + 1)
    y = global_rows[:, None] * dx
    x = global_cols[None, :] * dx

    phase = (ctx["instance_seed"] % 1000) / 1000.0

    u = np.zeros((PADDED_N, PADDED_N), dtype=np.float64)
    initial = np.sin(np.pi * y) * np.sin(np.pi * x)
    initial *= 1.0 + 0.05 * phase
    u[1:-1, 1:-1] = initial[1:-1, 1:-1]

    kappa = 1.0 + 0.25 * np.sin(2.0 * np.pi * x) * np.sin(2.0 * np.pi * y)
    kappa = np.ascontiguousarray(kappa, dtype=np.float64)

    source = np.zeros((PADDED_N, PADDED_N), dtype=np.float64)

    return u, kappa, source


@cuda.jit
def pack_halos_kernel(u, send_buffers):
    i = cuda.grid(1)

    if i < LOCAL_N:
        j = i + 1
        send_buffers[0, i] = u[1, j]
        send_buffers[1, i] = u[LOCAL_N, j]
        send_buffers[2, i] = u[j, 1]
        send_buffers[3, i] = u[j, LOCAL_N]


@cuda.jit
def unpack_halos_kernel(u, recv_buffers):
    i = cuda.grid(1)

    if i < LOCAL_N:
        j = i + 1
        u[0, j] = recv_buffers[0, i]
        u[LOCAL_N + 1, j] = recv_buffers[1, i]
        u[j, 0] = recv_buffers[2, i]
        u[j, LOCAL_N + 1] = recv_buffers[3, i]


'''

path = "hybrid_mpi_cuda_cache.py"

with open(path, "r", encoding="utf-8") as file:
    code = file.read()

marker = "def main():"
if "def initialize_fields(ctx):" in code:
    raise RuntimeError("Task 2 code is already present.")
if code.count(marker) != 1:
    raise RuntimeError("Could not locate a unique main() function.")

code = code.replace(marker, task2_code + marker, 1)

with open(path, "w", encoding="utf-8") as file:
    file.write(code)

print("Task 2 code added.")

Task 2 code added.


In [6]:
%%writefile test_task2.py
import sys

import numpy as np
from numba import cuda

from hybrid_mpi_cuda_cache import (
    LOCAL_N,
    PADDED_N,
    init_hybrid_context,
    initialize_fields,
    pack_halos_kernel,
    unpack_halos_kernel,
)


ctx = init_hybrid_context(sys.argv[1])

try:
    u, kappa, source = initialize_fields(ctx)

    assert u.shape == (PADDED_N, PADDED_N)
    assert kappa.shape == u.shape
    assert source.shape == u.shape
    assert np.all(np.isfinite(u))
    assert np.all(kappa > 0)

    d_u = cuda.to_device(u)
    d_send = cuda.device_array((4, LOCAL_N), dtype=np.float64)

    threads = 128
    blocks = (LOCAL_N + threads - 1) // threads

    pack_halos_kernel[blocks, threads](d_u, d_send)
    packed = d_send.copy_to_host()

    expected = np.stack([
        u[1, 1:-1],
        u[LOCAL_N, 1:-1],
        u[1:-1, 1],
        u[1:-1, LOCAL_N],
    ])

    np.testing.assert_array_equal(packed, expected)

    received = np.arange(4 * LOCAL_N, dtype=np.float64).reshape(4, LOCAL_N)
    d_recv = cuda.to_device(received)

    unpack_halos_kernel[blocks, threads](d_u, d_recv)
    unpacked = d_u.copy_to_host()

    np.testing.assert_array_equal(unpacked[0, 1:-1], received[0])
    np.testing.assert_array_equal(unpacked[-1, 1:-1], received[1])
    np.testing.assert_array_equal(unpacked[1:-1, 0], received[2])
    np.testing.assert_array_equal(unpacked[1:-1, -1], received[3])
    np.testing.assert_array_equal(unpacked[1:-1, 1:-1], u[1:-1, 1:-1])

    print(
        f"rank={ctx['rank']}: field and halo kernel tests passed",
        flush=True,
    )
finally:
    ctx["win"].Free()
    ctx["node_comm"].Free()
    ctx["cart_comm"].Free()

Overwriting test_task2.py


In [7]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "test_task2.py",
    GITHUB_USERNAME,
]

proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Task 2 test failed with code {proc.returncode}")

rank=0: field and halo kernel tests passed
rank=2: field and halo kernel tests passed
rank=3: field and halo kernel tests passed
rank=1: field and halo kernel tests passed



## Task 3 — Shared-Memory Tiled Stencil

In [8]:
task3_kernel_code = '''
from numba import float64

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2


@cuda.jit
def tiled_stencil_kernel(u, kappa, source, out, dt, dx, mode):
    # CUDA x corresponds to array columns; CUDA y corresponds to rows.
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    col = cuda.blockIdx.x * TILE_DIM + tx + 1
    row = cuda.blockIdx.y * TILE_DIM + ty + 1

    tile = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)

    if row <= LOCAL_N and col <= LOCAL_N:
        tile[ty + 1, tx + 1] = u[row, col]

    if tx == 0 and row <= LOCAL_N:
        tile[ty + 1, 0] = u[row, col - 1]

    if tx == TILE_DIM - 1 and row <= LOCAL_N and col <= LOCAL_N:
        tile[ty + 1, SHARED_DIM - 1] = u[row, col + 1]

    if ty == 0 and col <= LOCAL_N:
        tile[0, tx + 1] = u[row - 1, col]

    if ty == TILE_DIM - 1 and row <= LOCAL_N and col <= LOCAL_N:
        tile[SHARED_DIM - 1, tx + 1] = u[row + 1, col]

    cuda.syncthreads()

    if row <= LOCAL_N and col <= LOCAL_N:
        is_border = (
            row == 1 or row == LOCAL_N
            or col == 1 or col == LOCAL_N
        )

        # mode 0: deep interior, mode 1: border, mode 2: all cells.
        if mode == 2 or (mode == 0 and not is_border) or (mode == 1 and is_border):
            center = tile[ty + 1, tx + 1]
            north = tile[ty, tx + 1]
            south = tile[ty + 2, tx + 1]
            west = tile[ty + 1, tx]
            east = tile[ty + 1, tx + 2]

            kc = kappa[row, col]
            kn = 0.5 * (kc + kappa[row - 1, col])
            ks = 0.5 * (kc + kappa[row + 1, col])
            kw = 0.5 * (kc + kappa[row, col - 1])
            ke = 0.5 * (kc + kappa[row, col + 1])

            flux = (
                kn * (north - center)
                + ks * (south - center)
                + kw * (west - center)
                + ke * (east - center)
            )

            out[row, col] = center + dt * (flux / (dx * dx) + source[row, col])


'''

path = "hybrid_mpi_cuda_cache.py"

with open(path, "r", encoding="utf-8") as file:
    code = file.read()

marker = "def main():"
if "def tiled_stencil_kernel(" in code:
    raise RuntimeError("Task 3 kernel is already present.")
if code.count(marker) != 1:
    raise RuntimeError("Could not locate a unique main() function.")

code = code.replace(marker, task3_kernel_code + marker, 1)

with open(path, "w", encoding="utf-8") as file:
    file.write(code)

print("Task 3 stencil kernel added.")

Task 3 stencil kernel added.


In [9]:
import numpy as np
from numba import cuda

from hybrid_mpi_cuda_cache import (
    LOCAL_N,
    PADDED_N,
    TILE_DIM,
    tiled_stencil_kernel,
)

rng = np.random.default_rng(42)

u = rng.random((PADDED_N, PADDED_N))
kappa = 0.75 + 0.5 * rng.random((PADDED_N, PADDED_N))
source = rng.random((PADDED_N, PADDED_N))

dx = 1.0 / 513.0
dt = 0.1 * dx * dx / np.max(kappa)

center = u[1:-1, 1:-1]
kc = kappa[1:-1, 1:-1]

flux = (
    0.5 * (kc + kappa[:-2, 1:-1]) * (u[:-2, 1:-1] - center)
    + 0.5 * (kc + kappa[2:, 1:-1]) * (u[2:, 1:-1] - center)
    + 0.5 * (kc + kappa[1:-1, :-2]) * (u[1:-1, :-2] - center)
    + 0.5 * (kc + kappa[1:-1, 2:]) * (u[1:-1, 2:] - center)
)
expected = center + dt * (flux / (dx * dx) + source[1:-1, 1:-1])

d_u = cuda.to_device(u)
d_kappa = cuda.to_device(kappa)
d_source = cuda.to_device(source)

initial_out = np.full_like(u, -1.0)
d_full = cuda.to_device(initial_out)
d_split = cuda.to_device(initial_out)

threads = (TILE_DIM, TILE_DIM)
blocks = (
    (LOCAL_N + TILE_DIM - 1) // TILE_DIM,
    (LOCAL_N + TILE_DIM - 1) // TILE_DIM,
)

tiled_stencil_kernel[blocks, threads](
    d_u, d_kappa, d_source, d_full, dt, dx, 2
)
tiled_stencil_kernel[blocks, threads](
    d_u, d_kappa, d_source, d_split, dt, dx, 0
)
tiled_stencil_kernel[blocks, threads](
    d_u, d_kappa, d_source, d_split, dt, dx, 1
)

full = d_full.copy_to_host()
split = d_split.copy_to_host()

np.testing.assert_allclose(full[1:-1, 1:-1], expected, rtol=1e-13, atol=1e-13)
np.testing.assert_array_equal(split, full)

assert np.all(full[0, :] == -1.0)
assert np.all(full[-1, :] == -1.0)
assert np.all(full[:, 0] == -1.0)
assert np.all(full[:, -1] == -1.0)

print(f"Maximum CPU/GPU difference: {np.max(np.abs(full[1:-1, 1:-1] - expected)):.3e}")
print("Full and split stencil tests passed.")

Maximum CPU/GPU difference: 1.110e-16
Full and split stencil tests passed.


## Task 3.1 — Serialized MPI Halo Exchange

In [10]:
task3_exchange_code = '''
DIRECTIONS = ("up", "down", "left", "right")
OPPOSITE = (1, 0, 3, 2)
HALO_TAG_BASE = 100


def allocate_halo_buffers():
    return {
        "device_send": cuda.device_array((4, LOCAL_N), dtype=np.float64),
        "device_recv": cuda.device_array((4, LOCAL_N), dtype=np.float64),
        "host_send": cuda.pinned_array((4, LOCAL_N), dtype=np.float64),
        "host_recv": cuda.pinned_array((4, LOCAL_N), dtype=np.float64),
    }


def exchange_halos_serialized(ctx, d_u, buffers):
    cart = ctx["cart_comm"]
    neighbors = [ctx["neighbors"][name] for name in DIRECTIONS]

    threads = 128
    blocks = (LOCAL_N + threads - 1) // threads

    pack_halos_kernel[blocks, threads](d_u, buffers["device_send"])
    buffers["device_send"].copy_to_host(buffers["host_send"])

    # PROC_NULL receives leave the host buffer unchanged.
    buffers["host_recv"].fill(0.0)

    requests = []

    for direction, neighbor in enumerate(neighbors):
        requests.append(
            cart.Irecv(
                buffers["host_recv"][direction],
                source=neighbor,
                tag=HALO_TAG_BASE + OPPOSITE[direction],
            )
        )

    for direction, neighbor in enumerate(neighbors):
        requests.append(
            cart.Isend(
                buffers["host_send"][direction],
                dest=neighbor,
                tag=HALO_TAG_BASE + direction,
            )
        )

    MPI.Request.Waitall(requests)

    buffers["device_recv"].copy_to_device(buffers["host_recv"])
    unpack_halos_kernel[blocks, threads](d_u, buffers["device_recv"])
    cuda.synchronize()


'''

path = "hybrid_mpi_cuda_cache.py"

with open(path, "r", encoding="utf-8") as file:
    code = file.read()

marker = "def main():"
if "def exchange_halos_serialized(" in code:
    raise RuntimeError("Halo exchange code is already present.")
if code.count(marker) != 1:
    raise RuntimeError("Could not locate a unique main() function.")

code = code.replace(marker, task3_exchange_code + marker, 1)

with open(path, "w", encoding="utf-8") as file:
    file.write(code)

print("Serialized halo exchange added.")

Serialized halo exchange added.


In [11]:
%%writefile test_halo_exchange.py
import sys

import numpy as np
from mpi4py import MPI
from numba import cuda

from hybrid_mpi_cuda_cache import (
    PADDED_N,
    allocate_halo_buffers,
    exchange_halos_serialized,
    init_hybrid_context,
)


ctx = init_hybrid_context(sys.argv[1])

try:
    u = np.zeros((PADDED_N, PADDED_N), dtype=np.float64)
    u[1:-1, 1:-1] = ctx["rank"] + 1

    d_u = cuda.to_device(u)
    buffers = allocate_halo_buffers()

    exchange_halos_serialized(ctx, d_u, buffers)
    result = d_u.copy_to_host()

    slices = {
        "up": result[0, 1:-1],
        "down": result[-1, 1:-1],
        "left": result[1:-1, 0],
        "right": result[1:-1, -1],
    }

    for direction, values in slices.items():
        neighbor = ctx["neighbors"][direction]
        expected = 0 if neighbor == MPI.PROC_NULL else neighbor + 1
        np.testing.assert_array_equal(values, expected)

    np.testing.assert_array_equal(
        result[1:-1, 1:-1],
        np.full((256, 256), ctx["rank"] + 1, dtype=np.float64),
    )

    print(f"rank={ctx['rank']}: halo exchange test passed", flush=True)

finally:
    ctx["win"].Free()
    ctx["node_comm"].Free()
    ctx["cart_comm"].Free()

Overwriting test_halo_exchange.py


In [12]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "test_halo_exchange.py",
    GITHUB_USERNAME,
]

proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    print(f"Halo exchange test failed with code {proc.returncode}")


STDERR:
 
Halo exchange test failed with code 1


## Task 3.1 — Halo Exchange Diagnostics

In [13]:
import os
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--tag-output",
    "--mca", "orte_base_help_aggregate", "0",
    "-np", "4",
    "python3", "-u",
    "test_halo_exchange.py",
    GITHUB_USERNAME,
]

env = os.environ.copy()
env["PYTHONFAULTHANDLER"] = "1"

try:
    proc = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        timeout=120,
    )
    print("Return code:", proc.returncode)
    print("Combined output:")
    print(proc.stdout if proc.stdout else "<empty>")
except subprocess.TimeoutExpired as exc:
    print("Execution timed out.")
    print("Partial output:", exc.stdout)

Return code: 0
Combined output:
[1,3]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,3]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,0]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,0]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,2]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,2]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,1]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1

## Task 3.2 — Serialized and Overlapped Stencil Steps

In [14]:
task3_steps_code = '''
def step_serialized(ctx, d_u, d_out, d_kappa, d_source, dt, dx, buffers):
    exchange_halos_serialized(ctx, d_u, buffers)

    threads = (TILE_DIM, TILE_DIM)
    blocks = (
        (LOCAL_N + TILE_DIM - 1) // TILE_DIM,
        (LOCAL_N + TILE_DIM - 1) // TILE_DIM,
    )

    tiled_stencil_kernel[blocks, threads](
        d_u, d_kappa, d_source, d_out, dt, dx, 2
    )
    cuda.synchronize()


def step_overlapped(
    ctx, d_u, d_out, d_kappa, d_source, dt, dx, buffers,
    stream_interior, stream_boundary,
):
    cart = ctx["cart_comm"]
    neighbors = [ctx["neighbors"][name] for name in DIRECTIONS]

    halo_threads = 128
    halo_blocks = (LOCAL_N + halo_threads - 1) // halo_threads

    stencil_threads = (TILE_DIM, TILE_DIM)
    stencil_blocks = (
        (LOCAL_N + TILE_DIM - 1) // TILE_DIM,
        (LOCAL_N + TILE_DIM - 1) // TILE_DIM,
    )

    buffers["host_recv"].fill(0.0)
    requests = []

    for direction, neighbor in enumerate(neighbors):
        requests.append(
            cart.Irecv(
                buffers["host_recv"][direction],
                source=neighbor,
                tag=HALO_TAG_BASE + OPPOSITE[direction],
            )
        )

    tiled_stencil_kernel[stencil_blocks, stencil_threads, stream_interior](
        d_u, d_kappa, d_source, d_out, dt, dx, 0
    )

    pack_halos_kernel[halo_blocks, halo_threads, stream_boundary](
        d_u, buffers["device_send"]
    )
    buffers["device_send"].copy_to_host(
        buffers["host_send"], stream=stream_boundary
    )

    # MPI must not read the send buffers until the D2H copy completes.
    stream_boundary.synchronize()

    for direction, neighbor in enumerate(neighbors):
        requests.append(
            cart.Isend(
                buffers["host_send"][direction],
                dest=neighbor,
                tag=HALO_TAG_BASE + direction,
            )
        )

    MPI.Request.Waitall(requests)

    buffers["device_recv"].copy_to_device(
        buffers["host_recv"], stream=stream_boundary
    )
    unpack_halos_kernel[halo_blocks, halo_threads, stream_boundary](
        d_u, buffers["device_recv"]
    )
    tiled_stencil_kernel[stencil_blocks, stencil_threads, stream_boundary](
        d_u, d_kappa, d_source, d_out, dt, dx, 1
    )

    # Both output regions must be complete before the next step.
    stream_interior.synchronize()
    stream_boundary.synchronize()


'''

path = "hybrid_mpi_cuda_cache.py"

with open(path, "r", encoding="utf-8") as file:
    code = file.read()

marker = "def main():"
if "def step_overlapped(" in code:
    raise RuntimeError("Task 3 step functions are already present.")
if code.count(marker) != 1:
    raise RuntimeError("Could not locate a unique main() function.")

code = code.replace(marker, task3_steps_code + marker, 1)

with open(path, "w", encoding="utf-8") as file:
    file.write(code)

print("Serialized and overlapped step functions added.")

Serialized and overlapped step functions added.


In [15]:
%%writefile test_stencil_steps.py
import sys

import numpy as np
from mpi4py import MPI
from numba import cuda

from hybrid_mpi_cuda_cache import (
    allocate_halo_buffers,
    init_hybrid_context,
    initialize_fields,
    step_overlapped,
    step_serialized,
)


ctx = init_hybrid_context(sys.argv[1])

try:
    u, kappa, source = initialize_fields(ctx)

    dx = 1.0 / 513.0
    local_kappa_max = float(np.max(kappa))
    kappa_max = ctx["cart_comm"].allreduce(local_kappa_max, op=MPI.MAX)
    dt = 0.2 * dx * dx / kappa_max

    d_kappa = cuda.to_device(kappa)
    d_source = cuda.to_device(source)

    serial_u = cuda.to_device(u)
    serial_out = cuda.to_device(u)
    overlap_u = cuda.to_device(u)
    overlap_out = cuda.to_device(u)

    serial_buffers = allocate_halo_buffers()
    overlap_buffers = allocate_halo_buffers()

    stream_interior = cuda.stream()
    stream_boundary = cuda.stream()

    for _ in range(10):
        step_serialized(
            ctx, serial_u, serial_out, d_kappa, d_source,
            dt, dx, serial_buffers,
        )
        serial_u, serial_out = serial_out, serial_u

    ctx["cart_comm"].Barrier()

    for _ in range(10):
        step_overlapped(
            ctx, overlap_u, overlap_out, d_kappa, d_source,
            dt, dx, overlap_buffers, stream_interior, stream_boundary,
        )
        overlap_u, overlap_out = overlap_out, overlap_u

    serial_result = serial_u.copy_to_host()
    overlap_result = overlap_u.copy_to_host()

    local_diff = float(np.max(np.abs(
        serial_result[1:-1, 1:-1] - overlap_result[1:-1, 1:-1]
    )))
    global_diff = ctx["cart_comm"].allreduce(local_diff, op=MPI.MAX)

    print(f"rank={ctx['rank']}: local max difference={local_diff:.3e}", flush=True)

    if ctx["rank"] == 0:
        print(f"Global max difference: {global_diff:.3e}", flush=True)
        print(f"dt={dt:.8e}, CFL limit={dx * dx / (4.0 * kappa_max):.8e}", flush=True)

    assert global_diff < 1e-12

finally:
    ctx["win"].Free()
    ctx["node_comm"].Free()
    ctx["cart_comm"].Free()

Writing test_stencil_steps.py


In [16]:
import os
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--tag-output",
    "-np", "4",
    "python3", "-u",
    "test_stencil_steps.py",
    GITHUB_USERNAME,
]

env = os.environ.copy()
env["PYTHONFAULTHANDLER"] = "1"

proc = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
    timeout=180,
)

print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(f"Stencil step test failed with code {proc.returncode}")

[1,0]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,0]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,2]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,2]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,1]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,1]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,3]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,3]<stderr>:  warn(errors.NumbaP

## Task 3.3 — Prevent Concurrent Ghost-Cell Access

In [17]:
path = "hybrid_mpi_cuda_cache.py"

with open(path, "r", encoding="utf-8") as file:
    code = file.read()

replacements = {
    'if tx == 0 and row <= LOCAL_N:':
        'if tx == 0 and row <= LOCAL_N and (mode != 0 or col > 1):',

    'if tx == TILE_DIM - 1 and row <= LOCAL_N and col <= LOCAL_N:':
        'if tx == TILE_DIM - 1 and row <= LOCAL_N and col <= LOCAL_N and (mode != 0 or col < LOCAL_N):',

    'if ty == 0 and col <= LOCAL_N:':
        'if ty == 0 and col <= LOCAL_N and (mode != 0 or row > 1):',

    'if ty == TILE_DIM - 1 and row <= LOCAL_N and col <= LOCAL_N:':
        'if ty == TILE_DIM - 1 and row <= LOCAL_N and col <= LOCAL_N and (mode != 0 or row < LOCAL_N):',
}

for old, new in replacements.items():
    if code.count(old) != 1:
        raise RuntimeError(f"Expected exactly one occurrence: {old}")

for old, new in replacements.items():
    code = code.replace(old, new, 1)

with open(path, "w", encoding="utf-8") as file:
    file.write(code)

print("Deep-interior ghost-cell loads disabled.")

Deep-interior ghost-cell loads disabled.


In [18]:
import os
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--tag-output",
    "-np", "4",
    "python3", "-u",
    "test_stencil_steps.py",
    GITHUB_USERNAME,
]

env = os.environ.copy()
env["PYTHONFAULTHANDLER"] = "1"

proc = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
    timeout=180,
)

print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(f"Stencil step test failed with code {proc.returncode}")

[1,2]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,2]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,1]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,1]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,3]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,3]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,0]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,0]<stderr>:  warn(errors.NumbaP

## Task 4 — MPI Shared-Memory False-Sharing Benchmark

In [19]:
task4_code = '''
import time
from llvmlite import ir
from numba import njit, types
from numba.extending import intrinsic


@intrinsic
def atomic_increment_u64(typing_context, address):
    if address != types.uint64:
        return None

    signature = types.uint64(address)

    def codegen(context, builder, sig, args):
        pointer = builder.inttoptr(args[0], ir.IntType(64).as_pointer())
        one = ir.Constant(ir.IntType(64), 1)
        return builder.atomic_rmw("add", pointer, one, "monotonic")

    return signature, codegen


@njit(nogil=True)
def host_shm_accumulate(address, iterations):
    for _ in range(iterations):
        atomic_increment_u64(address)


def benchmark_shm_strides(ctx, iterations=40_000_000):
    node_comm = ctx["node_comm"]
    rank = ctx["node_rank"]
    view = ctx["shm_view"]
    win = ctx["win"]

    # Compile before timing.
    host_shm_accumulate(np.uint64(view.ctypes.data + rank * 8), 0)

    results = {}

    for stride in (1, 8, 16):
        if rank == 0:
            view.fill(0)

        win.Sync()
        node_comm.Barrier()
        win.Sync()

        address = np.uint64(view.ctypes.data + rank * stride * 8)

        node_comm.Barrier()
        start = time.perf_counter()
        host_shm_accumulate(address, iterations)
        elapsed = time.perf_counter() - start

        win.Sync()
        node_comm.Barrier()
        win.Sync()

        expected = np.uint64(iterations)
        if view[rank * stride] != expected:
            raise RuntimeError(
                f"Counter mismatch for rank={rank}, stride={stride}: "
                f"{view[rank * stride]} != {expected}"
            )

        max_elapsed = node_comm.allreduce(elapsed, op=MPI.MAX)
        results[stride] = max_elapsed

    return results


def audit_host_llvm():
    if not host_shm_accumulate.signatures:
        raise RuntimeError("Compile host_shm_accumulate before auditing.")

    signature = host_shm_accumulate.signatures[0]
    llvm_ir = host_shm_accumulate.inspect_llvm(signature)
    atomic_lines = [
        line.strip() for line in llvm_ir.splitlines()
        if "atomicrmw" in line
    ]

    if not atomic_lines:
        raise RuntimeError("LLVM audit failed: atomicrmw not found.")

    return llvm_ir, atomic_lines


'''

path = "hybrid_mpi_cuda_cache.py"

with open(path, "r", encoding="utf-8") as file:
    code = file.read()

marker = "def main():"
if "def host_shm_accumulate(" in code:
    raise RuntimeError("Task 4 code is already present.")
if code.count(marker) != 1:
    raise RuntimeError("Could not locate a unique main() function.")

code = code.replace(marker, task4_code + marker, 1)

with open(path, "w", encoding="utf-8") as file:
    file.write(code)

print("Task 4 benchmark code added.")

Task 4 benchmark code added.


In [20]:
%%writefile test_shm_benchmark.py
import sys

from hybrid_mpi_cuda_cache import (
    audit_host_llvm,
    benchmark_shm_strides,
    init_hybrid_context,
)


ctx = init_hybrid_context(sys.argv[1])

try:
    results = benchmark_shm_strides(ctx, iterations=10_000)
    llvm_ir, atomic_lines = audit_host_llvm()

    if ctx["rank"] == 0:
        print("Short benchmark results:", flush=True)
        for stride, elapsed in results.items():
            print(f"stride={stride}: {elapsed:.6f} s", flush=True)

        print(f"Static LLVM atomicrmw count: {len(atomic_lines)}", flush=True)
        for line in atomic_lines:
            print(line, flush=True)

        with open("host_shm_audit.ll", "w", encoding="utf-8") as file:
            file.write(llvm_ir)

        print("Full LLVM IR written to host_shm_audit.ll", flush=True)

finally:
    ctx["win"].Free()
    ctx["node_comm"].Free()
    ctx["cart_comm"].Free()

Writing test_shm_benchmark.py


In [21]:
import os
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--tag-output",
    "-np", "4",
    "python3", "-u",
    "test_shm_benchmark.py",
    GITHUB_USERNAME,
]

env = os.environ.copy()
env["PYTHONFAULTHANDLER"] = "1"

proc = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
    timeout=120,
)

print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(f"Shared-memory test failed with code {proc.returncode}")

[1,0]<stdout>:Short benchmark results:
[1,0]<stdout>:stride=1: 0.000172 s
[1,0]<stdout>:stride=8: 0.000150 s
[1,0]<stdout>:stride=16: 0.000123 s
[1,0]<stdout>:Static LLVM atomicrmw count: 9
[1,0]<stdout>:%.129.prol = atomicrmw add ptr %0, i64 1 monotonic, align 8[1,0]<stdout>:
[1,0]<stdout>:%.129 = atomicrmw add ptr %4, i64 1 monotonic, align 8[1,0]<stdout>:
[1,0]<stdout>:%.129.1 = atomicrmw add ptr %4, i64 1 monotonic, align 8
[1,0]<stdout>:%.129.2 = atomicrmw add ptr %4, i64 1 monotonic, align 8[1,0]<stdout>:
[1,0]<stdout>:%.129.3 = atomicrmw add ptr %4, i64 1 monotonic, align 8[1,0]<stdout>:
[1,0]<stdout>:%.129.4 = atomicrmw add ptr %4, i64 1 monotonic, align 8
[1,0]<stdout>:%.129.5 = atomicrmw add ptr %4, i64 1 monotonic, align 8
[1,0]<stdout>:%.129.6 = atomicrmw add ptr %4, i64 1 monotonic, align 8[1,0]<stdout>:
[1,0]<stdout>:%.129.7 = atomicrmw add ptr %4, i64 1 monotonic, align 8
[1,0]<stdout>:Full LLVM IR written to host_shm_audit.ll



## Task 5 — Verification and Manifest Export

In [22]:
task5_ptx_code = '''
import re


def audit_gpu_ptx():
    signatures = tiled_stencil_kernel.signatures

    if not signatures:
        raise RuntimeError("Compile tiled_stencil_kernel before auditing PTX.")

    ptx = tiled_stencil_kernel.inspect_asm(signatures[0])

    counts = {
        "st_shared_f64": len(re.findall(r"\\bst\\.shared\\.f64\\b", ptx)),
        "ld_shared_f64": len(re.findall(r"\\bld\\.shared\\.f64\\b", ptx)),
        "bar_sync": len(re.findall(r"\\bbar\\.sync\\b", ptx)),
        "st_shared_all": len(re.findall(r"\\bst\\.shared\\b", ptx)),
        "ld_shared_all": len(re.findall(r"\\bld\\.shared\\b", ptx)),
    }

    if counts["st_shared_all"] == 0:
        raise RuntimeError("PTX audit failed: shared-memory stores not found.")
    if counts["ld_shared_all"] == 0:
        raise RuntimeError("PTX audit failed: shared-memory loads not found.")
    if counts["bar_sync"] == 0:
        raise RuntimeError("PTX audit failed: bar.sync not found.")

    return ptx, counts


'''

path = "hybrid_mpi_cuda_cache.py"

with open(path, "r", encoding="utf-8") as file:
    code = file.read()

marker = "def main():"
if "def audit_gpu_ptx(" in code:
    raise RuntimeError("GPU PTX audit is already present.")
if code.count(marker) != 1:
    raise RuntimeError("Could not locate a unique main() function.")

code = code.replace(marker, task5_ptx_code + marker, 1)

with open(path, "w", encoding="utf-8") as file:
    file.write(code)

print("GPU PTX audit added.")

GPU PTX audit added.


In [23]:
%%writefile test_ptx_audit.py
import numpy as np
from numba import cuda

from hybrid_mpi_cuda_cache import (
    LOCAL_N,
    PADDED_N,
    TILE_DIM,
    audit_gpu_ptx,
    tiled_stencil_kernel,
)


u = np.zeros((PADDED_N, PADDED_N), dtype=np.float64)
kappa = np.ones_like(u)
source = np.zeros_like(u)

d_u = cuda.to_device(u)
d_kappa = cuda.to_device(kappa)
d_source = cuda.to_device(source)
d_out = cuda.to_device(u)

threads = (TILE_DIM, TILE_DIM)
blocks = (
    (LOCAL_N + TILE_DIM - 1) // TILE_DIM,
    (LOCAL_N + TILE_DIM - 1) // TILE_DIM,
)

tiled_stencil_kernel[blocks, threads](
    d_u, d_kappa, d_source, d_out, 1e-7, 1.0 / 513.0, 2
)
cuda.synchronize()

ptx, counts = audit_gpu_ptx()

for name, count in counts.items():
    print(f"{name}: {count}")

with open("stencil_audit.ptx", "w", encoding="utf-8") as file:
    file.write(ptx)

print("Full PTX written to stencil_audit.ptx")

Writing test_ptx_audit.py


In [24]:
import subprocess

proc = subprocess.run(
    ["python3", "-u", "test_ptx_audit.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=120,
)

print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(f"PTX audit failed with code {proc.returncode}")

st_shared_f64: 5
ld_shared_f64: 5
bar_sync: 1
st_shared_all: 5
ld_shared_all: 5
Full PTX written to stencil_audit.ptx



### Full Verification and Manifest

In [25]:
final_main_code = '''
import json

STENCIL_STEPS = 300
SHM_ITERS = 40_000_000


def global_energy(ctx, d_u, dx):
    host = d_u.copy_to_host()
    interior = host[1:-1, 1:-1]
    local_energy = float(np.sum(interior * interior) * dx * dx)
    return ctx["cart_comm"].allreduce(local_energy, op=MPI.SUM)


def run_simulation(ctx, mode, u, d_kappa, d_source, dt, dx):
    d_current = cuda.to_device(u)
    d_next = cuda.to_device(u)
    buffers = allocate_halo_buffers()

    stream_interior = cuda.stream()
    stream_boundary = cuda.stream()

    energies = {0: global_energy(ctx, d_current, dx)}

    ctx["cart_comm"].Barrier()
    start = MPI.Wtime()

    for step in range(1, STENCIL_STEPS + 1):
        if mode == "serialized":
            step_serialized(
                ctx, d_current, d_next, d_kappa, d_source,
                dt, dx, buffers,
            )
        elif mode == "overlapped":
            step_overlapped(
                ctx, d_current, d_next, d_kappa, d_source,
                dt, dx, buffers, stream_interior, stream_boundary,
            )
        else:
            raise ValueError(f"Unknown mode: {mode}")

        d_current, d_next = d_next, d_current

        if step in (100, 200, 300):
            energies[step] = global_energy(ctx, d_current, dx)

    elapsed = MPI.Wtime() - start
    max_elapsed = ctx["cart_comm"].allreduce(elapsed, op=MPI.MAX)

    return d_current.copy_to_host(), energies, max_elapsed * 1000.0


def main():
    if len(sys.argv) != 2:
        raise SystemExit(
            "Usage: python3 hybrid_mpi_cuda_cache.py GITHUB_USERNAME"
        )

    github_user = sys.argv[1].strip()
    ctx = init_hybrid_context(github_user)
    cart = ctx["cart_comm"]
    rank = ctx["rank"]

    try:
        u, kappa, source = initialize_fields(ctx)
        dx = 1.0 / (GLOBAL_N + 1)

        local_kappa_max = float(np.max(kappa))
        kappa_max = cart.allreduce(local_kappa_max, op=MPI.MAX)
        dt_max = dx * dx / (4.0 * kappa_max)
        dt = 0.2 * dx * dx / kappa_max

        if not dt <= dt_max:
            raise RuntimeError("CFL stability condition failed.")

        d_kappa = cuda.to_device(kappa)
        d_source = cuda.to_device(source)

        # Compile and warm up both execution paths outside the timed runs.
        warm_u = cuda.to_device(u)
        warm_out = cuda.to_device(u)
        warm_buffers = allocate_halo_buffers()
        stream_interior = cuda.stream()
        stream_boundary = cuda.stream()

        step_serialized(
            ctx, warm_u, warm_out, d_kappa, d_source,
            dt, dx, warm_buffers,
        )
        step_overlapped(
            ctx, warm_u, warm_out, d_kappa, d_source,
            dt, dx, warm_buffers, stream_interior, stream_boundary,
        )

        ptx, ptx_counts = audit_gpu_ptx()

        cart.Barrier()
        serial_field, serial_energy, serialized_ms = run_simulation(
            ctx, "serialized", u, d_kappa, d_source, dt, dx
        )

        cart.Barrier()
        overlap_field, overlap_energy, overlapped_ms = run_simulation(
            ctx, "overlapped", u, d_kappa, d_source, dt, dx
        )

        local_diff = float(np.max(np.abs(
            serial_field[1:-1, 1:-1] - overlap_field[1:-1, 1:-1]
        )))
        max_diff = cart.allreduce(local_diff, op=MPI.MAX)

        if max_diff >= 1e-12:
            raise RuntimeError(f"Parity verification failed: {max_diff}")

        for energies in (serial_energy, overlap_energy):
            values = [energies[s] for s in (0, 100, 200, 300)]
            if any(b > a + 1e-12 for a, b in zip(values, values[1:])):
                raise RuntimeError("Energy dissipation check failed.")

        shm_times = benchmark_shm_strides(ctx, iterations=SHM_ITERS)
        llvm_ir, atomic_lines = audit_host_llvm()

        manifest = {
            "github_username": github_user,
            "gpu_uuid": ctx["gpu_uuid"],
            "cpu_signature": ctx["cpu_signature"],
            "instance_seed": ctx["instance_seed"],
            "mpi_ranks": cart.Get_size(),
            "node_ranks": ctx["node_comm"].Get_size(),
            "global_shape": [GLOBAL_N, GLOBAL_N],
            "local_interior_shape": [LOCAL_N, LOCAL_N],
            "stencil_steps": STENCIL_STEPS,
            "dx": dx,
            "dt": dt,
            "dt_max": dt_max,
            "kappa_max": kappa_max,
            "max_diff": max_diff,
            "gpu_serialized_ms": serialized_ms,
            "gpu_overlapped_ms": overlapped_ms,
            "timing_scope": (
                "maximum rank wall time, including halo exchange, "
                "synchronization, and energy checkpoints"
            ),
            "energy_definition": "sum(u_interior ** 2) * dx ** 2",
            "energy_serialized": serial_energy,
            "energy_overlapped": overlap_energy,
            "shm_iterations": SHM_ITERS,
            "shm_operation": "atomic read-modify-write",
            "shm_stride_seconds": shm_times,
            "false_sharing_ratio_stride_1_over_8": (
                shm_times[1] / shm_times[8]
            ),
            "ptx_counts": ptx_counts,
            "llvm_static_atomicrmw_count": len(atomic_lines),
        }

        if rank == 0:
            print("PTX audit:", ptx_counts, flush=True)
            print(
                f"LLVM static atomicrmw count: {len(atomic_lines)}",
                flush=True,
            )
            for line in atomic_lines:
                print(line, flush=True)

            for step in (0, 100, 200, 300):
                print(
                    f"step={step}, "
                    f"energy_serialized={serial_energy[step]:.17e}, "
                    f"energy_overlapped={overlap_energy[step]:.17e}",
                    flush=True,
                )

            print(f"kappa_max={kappa_max:.17e}", flush=True)
            print(f"dt={dt:.17e}, dt_max={dt_max:.17e}", flush=True)
            print(f"max_diff={max_diff:.17e}", flush=True)
            print(f"gpu_serialized_ms={serialized_ms:.6f}", flush=True)
            print(f"gpu_overlapped_ms={overlapped_ms:.6f}", flush=True)

            for stride in (1, 8, 16):
                print(
                    f"shm_stride={stride}, "
                    f"max_seconds={shm_times[stride]:.9f}",
                    flush=True,
                )

            print(
                "false_sharing_ratio_stride_1_over_8="
                f"{shm_times[1] / shm_times[8]:.6f}",
                flush=True,
            )

            with open("hybrid_lab_manifest.json", "w", encoding="utf-8") as file:
                json.dump(manifest, file, indent=2)
                file.write("\\n")

            print("Manifest written to hybrid_lab_manifest.json", flush=True)

    finally:
        ctx["win"].Free()
        ctx["node_comm"].Free()
        ctx["cart_comm"].Free()


if __name__ == "__main__":
    main()
'''

path = "hybrid_mpi_cuda_cache.py"

with open(path, "r", encoding="utf-8") as file:
    code = file.read()

marker = "def main():"
if code.count(marker) != 1:
    raise RuntimeError("Could not locate a unique main() function.")

code = code[:code.index(marker)] + final_main_code

with open(path, "w", encoding="utf-8") as file:
    file.write(code)

print("Full verification main added.")

Full verification main added.


In [26]:
import os
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--tag-output",
    "-np", "4",
    "python3", "-u",
    "hybrid_mpi_cuda_cache.py",
    GITHUB_USERNAME,
]

env = os.environ.copy()
env["PYTHONFAULTHANDLER"] = "1"

try:
    proc = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        timeout=1800,
    )
    print(proc.stdout)

    if proc.returncode != 0:
        raise RuntimeError(f"Full run failed with code {proc.returncode}")
except subprocess.TimeoutExpired as exc:
    output = exc.stdout
    if isinstance(output, bytes):
        output = output.decode(errors="replace")
    print(output or "<no output>")
    raise RuntimeError("Full run exceeded the 30-minute timeout.")

[1,0]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,0]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,2]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,2]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,1]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,1]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))
[1,3]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
[1,3]<stderr>:  warn(errors.NumbaP